In [1]:
pip install gspread oauth2client pandas

In [ ]:
pip install tensorflow-datasets

In [3]:
import gspread
from oauth2client.service_account import ServiceAccountCredentials

def connect_sheet():
    scope = ["https://spreadsheets.google.com/feeds",
             "https://www.googleapis.com/auth/drive"]

    creds = ServiceAccountCredentials.from_json_keyfile_name(
        "credentials.json", scope)

    client = gspread.authorize(creds)
    sheet = client.open("contrastive_low_data_results").sheet1
    return sheet

def log_result(sheet, row):
    sheet.append_row(row)

In [11]:
import tensorflow as tf
import numpy as np

# =========================
# Config
# =========================
BATCH_SIZE = 128
EPOCHS = 20
LABEL_PERCENTS = [0.5]  # start small

np.random.seed(42)

# =========================
# Load Data (NO resize)
# =========================
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
y_train = y_train.flatten()
y_test = y_test.flatten()

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# =========================
# Subsample (numpy OK here)
# =========================
def get_subset(x, y, percent):
    n = len(x)
    k = int(n * percent)
    idx = np.random.choice(n, k, replace=False)
    return x[idx], y[idx]

# =========================
# TF Dataset (KEY FIX)
# =========================
def make_ds(x, y, train=True):
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if train:
        ds = ds.shuffle(10000)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

# =========================
# Lightweight CNN (FAST)
# =========================
def get_encoder():
    inputs = tf.keras.Input(shape=(32,32,3))
    x = tf.keras.layers.Conv2D(64, 3, activation='relu')(inputs)
    x = tf.keras.layers.MaxPool2D()(x)
    x = tf.keras.layers.Conv2D(128, 3, activation='relu')(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    return tf.keras.Model(inputs, x)

# =========================
# Classifier
# =========================
def get_classifier():
    enc = get_encoder()
    out = tf.keras.layers.Dense(10, activation='softmax')(enc.output)
    model = tf.keras.Model(enc.input, out)
    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

# =========================
# Contrastive Model
# =========================
def get_contrastive_model():
    enc = get_encoder()
    x = tf.keras.layers.Dense(128, activation='relu')(enc.output)
    z = tf.keras.layers.Dense(64)(x)
    return tf.keras.Model(enc.input, z)

# =========================
# Augment
# =========================
def augment(x):
    x = tf.image.random_flip_left_right(x)
    x = tf.image.random_brightness(x, 0.2)
    return x

# =========================
# Losses
# =========================
def simclr_loss(z1, z2, temp=0.5):
    z1 = tf.math.l2_normalize(z1, axis=1)
    z2 = tf.math.l2_normalize(z2, axis=1)

    sim = tf.matmul(z1, z2, transpose_b=True) / temp
    labels = tf.range(tf.shape(z1)[0])

    return tf.reduce_mean(
        tf.keras.losses.sparse_categorical_crossentropy(labels, sim, from_logits=True)
    )

def barlow_loss(z1, z2, lambd=0.005):
    z1 = (z1 - tf.reduce_mean(z1, axis=0)) / (tf.math.reduce_std(z1, axis=0)+1e-9)
    z2 = (z2 - tf.reduce_mean(z2, axis=0)) / (tf.math.reduce_std(z2, axis=0)+1e-9)

    c = tf.matmul(z1, z2, transpose_a=True) / tf.cast(tf.shape(z1)[0], tf.float32)

    on_diag = tf.reduce_sum((tf.linalg.diag_part(c) - 1) ** 2)
    off_diag = tf.reduce_sum(tf.square(c)) - tf.reduce_sum(tf.square(tf.linalg.diag_part(c)))

    return on_diag + lambd * off_diag

def supcon_loss(features, labels, temp=0.1):
    features = tf.math.l2_normalize(features, axis=1)
    logits = tf.matmul(features, features, transpose_b=True) / temp

    labels = tf.expand_dims(labels, 1)
    mask = tf.cast(tf.equal(labels, tf.transpose(labels)), tf.float32)

    exp_logits = tf.exp(logits)
    log_prob = logits - tf.math.log(tf.reduce_sum(exp_logits, axis=1, keepdims=True)+1e-9)

    return -tf.reduce_mean(
        tf.reduce_sum(mask * log_prob, axis=1) / (tf.reduce_sum(mask, axis=1)+1e-9)
    )

# =========================
# Train Contrastive
# =========================
def train_contrastive(model, ds, loss_fn):
    opt = tf.keras.optimizers.Adam()

    for epoch in range(EPOCHS):
        for x_batch, _ in ds:
            x1 = augment(x_batch)
            x2 = augment(x_batch)

            with tf.GradientTape() as tape:
                z1 = model(x1)
                z2 = model(x2)
                loss = loss_fn(z1, z2)

            grads = tape.gradient(loss, model.trainable_variables)
            opt.apply_gradients(zip(grads, model.trainable_variables))

# =========================
# Evaluate
# =========================
def evaluate_encoder(encoder, train_ds, test_ds):
    clf = tf.keras.Sequential([
        encoder,
        tf.keras.layers.Dense(10, activation='softmax')
    ])
    clf.compile(optimizer='adam',
                loss='sparse_categorical_crossentropy',
                metrics=['accuracy'])

    clf.fit(train_ds, epochs=3, verbose=0)
    _, acc = clf.evaluate(test_ds, verbose=0)
    return acc

# =========================
# Run
# =========================
results = []

for p in LABEL_PERCENTS:
    xs, ys = get_subset(x_train, y_train, p)

    train_ds = make_ds(xs, ys, True)
    test_ds = make_ds(x_test, y_test, False)

    # Supervised
    model = get_classifier()
    model.fit(train_ds, epochs=EPOCHS, verbose=0)
    _, acc = model.evaluate(test_ds, verbose=0)

    print("CIFAR10 | Supervised |", p, "|", acc, "|", EPOCHS)
    results.append(["CIFAR10", "Supervised", p, acc, EPOCHS, ""])

    # SimCLR
    model = get_contrastive_model()
    train_contrastive(model, train_ds, simclr_loss)
    acc = evaluate_encoder(model, train_ds, test_ds)

    print("CIFAR10 | SimCLR |", p, "|", acc, "|", EPOCHS)
    results.append(["CIFAR10", "SimCLR", p, acc, EPOCHS, ""])

    # Barlow Twins
    model = get_contrastive_model()
    train_contrastive(model, train_ds, barlow_loss)
    acc = evaluate_encoder(model, train_ds, test_ds)

    print("CIFAR10 | BarlowTwins |", p, "|", acc, "|", EPOCHS)
    results.append(["CIFAR10", "BarlowTwins", p, acc, EPOCHS, ""])

    # SupCon
    model = get_contrastive_model()
    opt = tf.keras.optimizers.Adam()

    for epoch in range(EPOCHS):
        for x_batch, y_batch in train_ds:
            with tf.GradientTape() as tape:
                z = model(x_batch)
                loss = supcon_loss(z, y_batch)

            grads = tape.gradient(loss, model.trainable_variables)
            opt.apply_gradients(zip(grads, model.trainable_variables))

    acc = evaluate_encoder(model, train_ds, test_ds)

    print("CIFAR10 | SupCon |", p, "|", acc, "|", EPOCHS)
    results.append(["CIFAR10", "SupCon", p, acc, EPOCHS, ""])

# =========================
# Final Table
# =========================

print("\nFINAL RESULTS:")
print("Dataset | Method | Label_Percent | Accuracy | Epochs | Notes")

for r in results:
    print(" | ".join(map(str, r)))




CIFAR10 | Supervised | 0.5 | 0.5109999775886536 | 20
CIFAR10 | SimCLR | 0.5 | 0.4124999940395355 | 20
CIFAR10 | BarlowTwins | 0.5 | 0.4320000112056732 | 20
CIFAR10 | SupCon | 0.5 | 0.5329999923706055 | 20

FINAL RESULTS:
Dataset | Method | Label_Percent | Accuracy | Epochs | Notes
CIFAR10 | Supervised | 0.5 | 0.5109999775886536 | 20 | 
CIFAR10 | SimCLR | 0.5 | 0.4124999940395355 | 20 | 
CIFAR10 | BarlowTwins | 0.5 | 0.4320000112056732 | 20 | 
CIFAR10 | SupCon | 0.5 | 0.5329999923706055 | 20 | 


In [12]:
import tensorflow as tf
import numpy as np

# =========================
# Config
# =========================


BATCH_SIZE = 128
EPOCHS = 20
LABEL_PERCENTS = [0.5]

np.random.seed(42)

# =========================
# Load CIFAR-100
# =========================


(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar100.load_data()
y_train = y_train.flatten()
y_test = y_test.flatten()

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# =========================
# Subsample
# =========================


def get_subset(x, y, percent):
    n = len(x)
    k = int(n * percent)
    idx = np.random.choice(n, k, replace=False)
    return x[idx], y[idx]

# =========================
# TF Dataset
# =========================


def make_ds(x, y, train=True):
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if train:
        ds = ds.shuffle(10000)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

# =========================
# Encoder
# =========================


def get_encoder():
    inputs = tf.keras.Input(shape=(32,32,3))
    x = tf.keras.layers.Conv2D(64, 3, activation='relu')(inputs)
    x = tf.keras.layers.MaxPool2D()(x)
    x = tf.keras.layers.Conv2D(128, 3, activation='relu')(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    return tf.keras.Model(inputs, x)

# =========================
# Classifier (100 classes)
# =========================


def get_classifier():
    enc = get_encoder()
    out = tf.keras.layers.Dense(100, activation='softmax')(enc.output)
    model = tf.keras.Model(enc.input, out)

    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

# =========================
# Contrastive Model
# =========================


def get_contrastive_model():
    enc = get_encoder()
    x = tf.keras.layers.Dense(128, activation='relu')(enc.output)
    z = tf.keras.layers.Dense(64)(x)
    return tf.keras.Model(enc.input, z)

# =========================
# Augment
# =========================


def augment(x):
    x = tf.image.random_flip_left_right(x)
    x = tf.image.random_brightness(x, 0.2)
    return x

# =========================
# Losses
# =========================


def simclr_loss(z1, z2, temp=0.5):
    z1 = tf.math.l2_normalize(z1, axis=1)
    z2 = tf.math.l2_normalize(z2, axis=1)

    sim = tf.matmul(z1, z2, transpose_b=True) / temp
    labels = tf.range(tf.shape(z1)[0])

    return tf.reduce_mean(
        tf.keras.losses.sparse_categorical_crossentropy(labels, sim, from_logits=True)
    )

def barlow_loss(z1, z2, lambd=0.005):
    z1 = (z1 - tf.reduce_mean(z1, axis=0)) / (tf.math.reduce_std(z1, axis=0)+1e-9)
    z2 = (z2 - tf.reduce_mean(z2, axis=0)) / (tf.math.reduce_std(z2, axis=0)+1e-9)

    c = tf.matmul(z1, z2, transpose_a=True) / tf.cast(tf.shape(z1)[0], tf.float32)

    on_diag = tf.reduce_sum((tf.linalg.diag_part(c) - 1) ** 2)
    off_diag = tf.reduce_sum(tf.square(c)) - tf.reduce_sum(tf.square(tf.linalg.diag_part(c)))

    return on_diag + lambd * off_diag

def supcon_loss(features, labels, temp=0.1):
    features = tf.math.l2_normalize(features, axis=1)
    logits = tf.matmul(features, features, transpose_b=True) / temp

    labels = tf.expand_dims(labels, 1)
    mask = tf.cast(tf.equal(labels, tf.transpose(labels)), tf.float32)

    exp_logits = tf.exp(logits)
    log_prob = logits - tf.math.log(tf.reduce_sum(exp_logits, axis=1, keepdims=True)+1e-9)

    return -tf.reduce_mean(
        tf.reduce_sum(mask * log_prob, axis=1) / (tf.reduce_sum(mask, axis=1)+1e-9)
    )

# =========================
# Train Contrastive
# =========================


def train_contrastive(model, ds, loss_fn):
    opt = tf.keras.optimizers.Adam()

    for epoch in range(EPOCHS):
        for x_batch, _ in ds:
            x1 = augment(x_batch)
            x2 = augment(x_batch)

            with tf.GradientTape() as tape:
                z1 = model(x1)
                z2 = model(x2)
                loss = loss_fn(z1, z2)

            grads = tape.gradient(loss, model.trainable_variables)
            opt.apply_gradients(zip(grads, model.trainable_variables))

# =========================
# Evaluate
# =========================


def evaluate_encoder(encoder, train_ds, test_ds):
    clf = tf.keras.Sequential([
        encoder,
        tf.keras.layers.Dense(100, activation='softmax')
    ])

    clf.compile(optimizer='adam',
                loss='sparse_categorical_crossentropy',
                metrics=['accuracy'])

    clf.fit(train_ds, epochs=3, verbose=0)
    _, acc = clf.evaluate(test_ds, verbose=0)
    return acc

# =========================
# Run
# =========================


results = []

for p in LABEL_PERCENTS:
    xs, ys = get_subset(x_train, y_train, p)

    train_ds = make_ds(xs, ys, True)
    test_ds = make_ds(x_test, y_test, False)

    # Supervised
    model = get_classifier()
    model.fit(train_ds, epochs=EPOCHS, verbose=0)
    _, acc = model.evaluate(test_ds, verbose=0)

    print("CIFAR100 | Supervised |", p, "|", acc, "|", EPOCHS)
    results.append(["CIFAR100", "Supervised", p, acc, EPOCHS, ""])

    # SimCLR
    model = get_contrastive_model()
    train_contrastive(model, train_ds, simclr_loss)
    acc = evaluate_encoder(model, train_ds, test_ds)

    print("CIFAR100 | SimCLR |", p, "|", acc, "|", EPOCHS)
    results.append(["CIFAR100", "SimCLR", p, acc, EPOCHS, ""])

    # Barlow Twins
    model = get_contrastive_model()
    train_contrastive(model, train_ds, barlow_loss)
    acc = evaluate_encoder(model, train_ds, test_ds)

    print("CIFAR100 | BarlowTwins |", p, "|", acc, "|", EPOCHS)
    results.append(["CIFAR100", "BarlowTwins", p, acc, EPOCHS, ""])

    # SupCon
    model = get_contrastive_model()
    opt = tf.keras.optimizers.Adam()

    for epoch in range(EPOCHS):
        for x_batch, y_batch in train_ds:
            with tf.GradientTape() as tape:
                z = model(x_batch)
                loss = supcon_loss(z, y_batch)

            grads = tape.gradient(loss, model.trainable_variables)
            opt.apply_gradients(zip(grads, model.trainable_variables))

    acc = evaluate_encoder(model, train_ds, test_ds)

    print("CIFAR100 | SupCon |", p, "|", acc, "|", EPOCHS)
    results.append(["CIFAR100", "SupCon", p, acc, EPOCHS, ""])

# =========================
# Final Table
# =========================


print("\nFINAL RESULTS:")
print("Dataset | Method | Label_Percent | Accuracy | Epochs | Notes")

for r in results:
    print(" | ".join(map(str, r)))



CIFAR100 | Supervised | 0.5 | 0.18080000579357147 | 20
CIFAR100 | SimCLR | 0.5 | 0.12099999934434891 | 20
CIFAR100 | BarlowTwins | 0.5 | 0.11569999903440475 | 20
CIFAR100 | SupCon | 0.5 | 0.17679999768733978 | 20

FINAL RESULTS:
Dataset | Method | Label_Percent | Accuracy | Epochs | Notes
CIFAR100 | Supervised | 0.5 | 0.18080000579357147 | 20 | 
CIFAR100 | SimCLR | 0.5 | 0.12099999934434891 | 20 | 
CIFAR100 | BarlowTwins | 0.5 | 0.11569999903440475 | 20 | 
CIFAR100 | SupCon | 0.5 | 0.17679999768733978 | 20 | 


In [13]:
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np

# =========================
# Config
# =========================
BATCH_SIZE = 64   # smaller (images bigger)
EPOCHS = 20
IMG_SIZE = 96
LABEL_PERCENTS = [0.5]

np.random.seed(42)

# =========================
# Load Dataset (TFDS)
# =========================
ds_train, ds_test = tfds.load(
    "caltech101",
    split=["train[:80%]", "train[80%:]"],
    as_supervised=True
)

NUM_CLASSES = 102  # Caltech101 has ~102 classes

# =========================
# Preprocess
# =========================
def preprocess(x, y):
    x = tf.image.resize(x, (IMG_SIZE, IMG_SIZE))
    x = tf.cast(x, tf.float32) / 255.0
    return x, y

ds_train = ds_train.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
ds_test = ds_test.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)

# Convert to numpy (only once)
x_train, y_train = [], []
for x, y in tfds.as_numpy(ds_train):
    x_train.append(x)
    y_train.append(y)

x_test, y_test = [], []
for x, y in tfds.as_numpy(ds_test):
    x_test.append(x)
    y_test.append(y)

x_train = np.array(x_train)
y_train = np.array(y_train)
x_test = np.array(x_test)
y_test = np.array(y_test)

# =========================
# Subsample
# =========================
def get_subset(x, y, percent):
    n = len(x)
    k = int(n * percent)
    idx = np.random.choice(n, k, replace=False)
    return x[idx], y[idx]

# =========================
# Dataset pipeline
# =========================
def make_ds(x, y, train=True):
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if train:
        ds = ds.shuffle(2000)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

# =========================
# Encoder (slightly deeper)
# =========================
def get_encoder():
    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = tf.keras.layers.Conv2D(64, 3, activation='relu')(inputs)
    x = tf.keras.layers.MaxPool2D()(x)
    x = tf.keras.layers.Conv2D(128, 3, activation='relu')(x)
    x = tf.keras.layers.MaxPool2D()(x)
    x = tf.keras.layers.Conv2D(256, 3, activation='relu')(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    return tf.keras.Model(inputs, x)

# =========================
# Classifier
# =========================
def get_classifier():
    enc = get_encoder()
    out = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')(enc.output)
    model = tf.keras.Model(enc.input, out)

    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

# =========================
# Contrastive model
# =========================
def get_contrastive_model():
    enc = get_encoder()
    x = tf.keras.layers.Dense(128, activation='relu')(enc.output)
    z = tf.keras.layers.Dense(64)(x)
    return tf.keras.Model(enc.input, z)

# =========================
# Augment
# =========================
def augment(x):
    x = tf.image.random_flip_left_right(x)
    x = tf.image.random_brightness(x, 0.2)
    return x

# =========================
# Losses
# =========================
def simclr_loss(z1, z2, temp=0.5):
    z1 = tf.math.l2_normalize(z1, axis=1)
    z2 = tf.math.l2_normalize(z2, axis=1)

    sim = tf.matmul(z1, z2, transpose_b=True) / temp
    labels = tf.range(tf.shape(z1)[0])

    return tf.reduce_mean(
        tf.keras.losses.sparse_categorical_crossentropy(labels, sim, from_logits=True)
    )

# =========================
# Train contrastive
# =========================
def train_contrastive(model, ds):
    opt = tf.keras.optimizers.Adam()

    for epoch in range(EPOCHS):
        for x_batch, _ in ds:
            x1 = augment(x_batch)
            x2 = augment(x_batch)

            with tf.GradientTape() as tape:
                z1 = model(x1)
                z2 = model(x2)
                loss = simclr_loss(z1, z2)

            grads = tape.gradient(loss, model.trainable_variables)
            opt.apply_gradients(zip(grads, model.trainable_variables))

# =========================
# Evaluate
# =========================
def evaluate_encoder(encoder, train_ds, test_ds):
    clf = tf.keras.Sequential([
        encoder,
        tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')
    ])

    clf.compile(optimizer='adam',
                loss='sparse_categorical_crossentropy',
                metrics=['accuracy'])

    clf.fit(train_ds, epochs=3, verbose=0)
    _, acc = clf.evaluate(test_ds, verbose=0)
    return acc

# =========================
# Run Experiments
# =========================
results = []

for p in LABEL_PERCENTS:
    xs, ys = get_subset(x_train, y_train, p)

    train_ds = make_ds(xs, ys, True)
    test_ds = make_ds(x_test, y_test, False)

    # Supervised
    model = get_classifier()
    model.fit(train_ds, epochs=EPOCHS, verbose=0)
    _, acc = model.evaluate(test_ds, verbose=0)

    print("Caltech101 | Supervised |", p, "|", acc, "|", EPOCHS)
    results.append(["Caltech101", "Supervised", p, acc, EPOCHS, ""])

    # SimCLR
    model = get_contrastive_model()
    train_contrastive(model, train_ds)
    acc = evaluate_encoder(model, train_ds, test_ds)

    print("Caltech101 | SimCLR |", p, "|", acc, "|", EPOCHS)
    results.append(["Caltech101", "SimCLR", p, acc, EPOCHS, ""])

# =========================
# Final Table
# =========================
print("\nFINAL RESULTS:")
print("Dataset | Method | Label_Percent | Accuracy | Epochs | Notes")

for r in results:
    print(" | ".join(map(str, r)))



Caltech101 | Supervised | 0.5 | 0.08496732264757156 | 20
Caltech101 | SimCLR | 0.5 | 0.03267974033951759 | 20

FINAL RESULTS:
Dataset | Method | Label_Percent | Accuracy | Epochs | Notes
Caltech101 | Supervised | 0.5 | 0.08496732264757156 | 20 | 
Caltech101 | SimCLR | 0.5 | 0.03267974033951759 | 20 | 


In [14]:
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np

# =========================
# Config
# =========================
BATCH_SIZE = 64
EPOCHS = 20
IMG_SIZE = 96
LABEL_PERCENTS = [0.5]

np.random.seed(42)

# =========================
# Load dataset
# =========================
ds_train, ds_test = tfds.load(
    "oxford_flowers102",
    split=["train", "test"],
    as_supervised=True
)

NUM_CLASSES = 102

# =========================
# Preprocess
# =========================
def preprocess(x, y):
    x = tf.image.resize(x, (IMG_SIZE, IMG_SIZE))
    x = tf.cast(x, tf.float32) / 255.0
    return x, y

ds_train = ds_train.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
ds_test = ds_test.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)

# Convert to numpy (dataset is small enough)
x_train, y_train = [], []
for x, y in tfds.as_numpy(ds_train):
    x_train.append(x)
    y_train.append(y)

x_test, y_test = [], []
for x, y in tfds.as_numpy(ds_test):
    x_test.append(x)
    y_test.append(y)

x_train = np.array(x_train)
y_train = np.array(y_train)
x_test = np.array(x_test)
y_test = np.array(y_test)

# =========================
# Subsample
# =========================
def get_subset(x, y, percent):
    n = len(x)
    k = int(n * percent)
    idx = np.random.choice(n, k, replace=False)
    return x[idx], y[idx]

# =========================
# Dataset pipeline
# =========================
def make_ds(x, y, train=True):
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if train:
        ds = ds.shuffle(2000)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

# =========================
# Encoder
# =========================
def get_encoder():
    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = tf.keras.layers.Conv2D(64, 3, activation='relu')(inputs)
    x = tf.keras.layers.MaxPool2D()(x)
    x = tf.keras.layers.Conv2D(128, 3, activation='relu')(x)
    x = tf.keras.layers.MaxPool2D()(x)
    x = tf.keras.layers.Conv2D(256, 3, activation='relu')(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    return tf.keras.Model(inputs, x)

# =========================
# Classifier
# =========================
def get_classifier():
    enc = get_encoder()
    out = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')(enc.output)
    model = tf.keras.Model(enc.input, out)

    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

# =========================
# Contrastive model
# =========================
def get_contrastive_model():
    enc = get_encoder()
    x = tf.keras.layers.Dense(128, activation='relu')(enc.output)
    z = tf.keras.layers.Dense(64)(x)
    return tf.keras.Model(enc.input, z)

# =========================
# Augmentation
# =========================
def augment(x):
    x = tf.image.random_flip_left_right(x)
    x = tf.image.random_brightness(x, 0.2)
    return x

# =========================
# Losses
# =========================
def simclr_loss(z1, z2, temp=0.5):
    z1 = tf.math.l2_normalize(z1, axis=1)
    z2 = tf.math.l2_normalize(z2, axis=1)

    sim = tf.matmul(z1, z2, transpose_b=True) / temp
    labels = tf.range(tf.shape(z1)[0])

    return tf.reduce_mean(
        tf.keras.losses.sparse_categorical_crossentropy(labels, sim, from_logits=True)
    )

def barlow_loss(z1, z2, lambd=0.005):
    z1 = (z1 - tf.reduce_mean(z1, axis=0)) / (tf.math.reduce_std(z1, axis=0)+1e-9)
    z2 = (z2 - tf.reduce_mean(z2, axis=0)) / (tf.math.reduce_std(z2, axis=0)+1e-9)

    c = tf.matmul(z1, z2, transpose_a=True) / tf.cast(tf.shape(z1)[0], tf.float32)

    on_diag = tf.reduce_sum((tf.linalg.diag_part(c) - 1) ** 2)
    off_diag = tf.reduce_sum(tf.square(c)) - tf.reduce_sum(tf.square(tf.linalg.diag_part(c)))

    return on_diag + lambd * off_diag

def supcon_loss(features, labels, temp=0.1):
    features = tf.math.l2_normalize(features, axis=1)
    logits = tf.matmul(features, features, transpose_b=True) / temp

    labels = tf.expand_dims(labels, 1)
    mask = tf.cast(tf.equal(labels, tf.transpose(labels)), tf.float32)

    exp_logits = tf.exp(logits)
    log_prob = logits - tf.math.log(tf.reduce_sum(exp_logits, axis=1, keepdims=True)+1e-9)

    return -tf.reduce_mean(
        tf.reduce_sum(mask * log_prob, axis=1) / (tf.reduce_sum(mask, axis=1)+1e-9)
    )

# =========================
# Train contrastive
# =========================
def train_contrastive(model, ds, loss_fn):
    opt = tf.keras.optimizers.Adam()

    for epoch in range(EPOCHS):
        for x_batch, _ in ds:
            x1 = augment(x_batch)
            x2 = augment(x_batch)

            with tf.GradientTape() as tape:
                z1 = model(x1)
                z2 = model(x2)
                loss = loss_fn(z1, z2)

            grads = tape.gradient(loss, model.trainable_variables)
            opt.apply_gradients(zip(grads, model.trainable_variables))

# =========================
# Evaluate
# =========================
def evaluate_encoder(encoder, train_ds, test_ds):
    clf = tf.keras.Sequential([
        encoder,
        tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')
    ])

    clf.compile(optimizer='adam',
                loss='sparse_categorical_crossentropy',
                metrics=['accuracy'])

    clf.fit(train_ds, epochs=3, verbose=0)
    _, acc = clf.evaluate(test_ds, verbose=0)
    return acc

# =========================
# Run experiments
# =========================
results = []

for p in LABEL_PERCENTS:
    xs, ys = get_subset(x_train, y_train, p)

    train_ds = make_ds(xs, ys, True)
    test_ds = make_ds(x_test, y_test, False)

    # Supervised
    model = get_classifier()
    model.fit(train_ds, epochs=EPOCHS, verbose=0)
    _, acc = model.evaluate(test_ds, verbose=0)

    print("Flowers102 | Supervised |", p, "|", acc, "|", EPOCHS)
    results.append(["Flowers102", "Supervised", p, acc, EPOCHS, ""])

    # SimCLR
    model = get_contrastive_model()
    train_contrastive(model, train_ds, simclr_loss)
    acc = evaluate_encoder(model, train_ds, test_ds)

    print("Flowers102 | SimCLR |", p, "|", acc, "|", EPOCHS)
    results.append(["Flowers102", "SimCLR", p, acc, EPOCHS, ""])

    # Barlow Twins
    model = get_contrastive_model()
    train_contrastive(model, train_ds, barlow_loss)
    acc = evaluate_encoder(model, train_ds, test_ds)

    print("Flowers102 | BarlowTwins |", p, "|", acc, "|", EPOCHS)
    results.append(["Flowers102", "BarlowTwins", p, acc, EPOCHS, ""])

    # SupCon
    model = get_contrastive_model()
    opt = tf.keras.optimizers.Adam()

    for epoch in range(EPOCHS):
        for x_batch, y_batch in train_ds:
            with tf.GradientTape() as tape:
                z = model(x_batch)
                loss = supcon_loss(z, y_batch)

            grads = tape.gradient(loss, model.trainable_variables)
            opt.apply_gradients(zip(grads, model.trainable_variables))

    acc = evaluate_encoder(model, train_ds, test_ds)

    print("Flowers102 | SupCon |", p, "|", acc, "|", EPOCHS)
    results.append(["Flowers102", "SupCon", p, acc, EPOCHS, ""])

# =========================
# Final output
# =========================
print("\nFINAL RESULTS:")
print("Dataset | Method | Label_Percent | Accuracy | Epochs | Notes")

for r in results:
    print(" | ".join(map(str, r)))




Flowers102 | Supervised | 0.5 | 0.1380712240934372 | 20
Flowers102 | SimCLR | 0.5 | 0.014799154363572598 | 20
Flowers102 | BarlowTwins | 0.5 | 0.039030738174915314 | 20
Flowers102 | SupCon | 0.5 | 0.027484143152832985 | 20

FINAL RESULTS:
Dataset | Method | Label_Percent | Accuracy | Epochs | Notes
Flowers102 | Supervised | 0.5 | 0.1380712240934372 | 20 | 
Flowers102 | SimCLR | 0.5 | 0.014799154363572598 | 20 | 
Flowers102 | BarlowTwins | 0.5 | 0.039030738174915314 | 20 | 
Flowers102 | SupCon | 0.5 | 0.027484143152832985 | 20 | 


In [15]:
import tensorflow as tf
import numpy as np
from collections import defaultdict

# =========================
# Config
# =========================
BATCH_SIZE = 128
EPOCHS = 5

LABEL_PERCENTS = [0.01, 0.05, 0.1, 0.5]
TEMPS = [0.05, 0.1]
ALPHAS = [0.5, 1.0]

np.random.seed(42)

# =========================
# Load dataset
# =========================
def load_dataset(name):
    if name == "CIFAR10":
        (x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
        num_classes = 10
    else:
        (x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar100.load_data()
        num_classes = 100

    y_train = y_train.flatten()
    y_test = y_test.flatten()

    x_train = x_train.astype("float32") / 255.0
    x_test = x_test.astype("float32") / 255.0

    return x_train, y_train, x_test, y_test, num_classes

# =========================
# Subsample
# =========================
def get_subset(x, y, percent):
    n = len(x)
    k = int(n * percent)
    idx = np.random.choice(n, k, replace=False)
    return x[idx], y[idx]

# =========================
# Dataset
# =========================
def make_ds(x, y, train=True):
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if train:
        ds = ds.shuffle(10000)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# =========================
# SAME CNN
# =========================
def get_encoder():
    inputs = tf.keras.Input(shape=(32,32,3))
    x = tf.keras.layers.Conv2D(64, 3, activation='relu')(inputs)
    x = tf.keras.layers.MaxPool2D()(x)
    x = tf.keras.layers.Conv2D(128, 3, activation='relu')(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    return tf.keras.Model(inputs, x)

def get_model():
    enc = get_encoder()
    x = tf.keras.layers.Dense(128, activation='relu')(enc.output)
    z = tf.keras.layers.Dense(64)(x)
    return tf.keras.Model(enc.input, z)

# =========================
# Augment
# =========================
def augment(x):
    x = tf.image.random_flip_left_right(x)
    x = tf.image.random_brightness(x, 0.2)
    return x

# =========================
# APW-SupCon Loss (FIXED)
# =========================
def apw_supcon_loss(z, y, temp, alpha):
    z = tf.math.l2_normalize(z, axis=1)

    sim = tf.matmul(z, z, transpose_b=True) / temp

    y = tf.expand_dims(y, 1)
    mask = tf.cast(tf.equal(y, tf.transpose(y)), tf.float32)

    # remove self-pairs
    mask = mask - tf.eye(tf.shape(mask)[0])

    class_counts = tf.reduce_sum(mask, axis=1)

    weights = tf.pow(class_counts + 1e-6, -alpha)

    exp_sim = tf.exp(sim)
    log_prob = sim - tf.math.log(tf.reduce_sum(exp_sim, axis=1, keepdims=True) + 1e-9)

    loss = -tf.reduce_mean(
        weights * tf.reduce_sum(mask * log_prob, axis=1) / (class_counts + 1e-9)
    )

    return loss

# =========================
# Train (2 views)
# =========================
def train(model, ds, temp, alpha):
    opt = tf.keras.optimizers.Adam()

    for epoch in range(EPOCHS):
        for x_batch, y_batch in ds:

            x1 = augment(x_batch)
            x2 = augment(x_batch)

            with tf.GradientTape() as tape:
                z1 = model(x1)
                z2 = model(x2)

                z = tf.concat([z1, z2], axis=0)
                y = tf.concat([y_batch, y_batch], axis=0)

                loss = apw_supcon_loss(z, y, temp, alpha)

            grads = tape.gradient(loss, model.trainable_variables)
            opt.apply_gradients(zip(grads, model.trainable_variables))

# =========================
# Evaluate (linear probe)
# =========================
def evaluate(encoder, train_ds, test_ds, num_classes):
    clf = tf.keras.Sequential([
        encoder,
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])

    clf.compile(optimizer='adam',
                loss='sparse_categorical_crossentropy',
                metrics=['accuracy'])

    clf.fit(train_ds, epochs=3, verbose=0)
    _, acc = clf.evaluate(test_ds, verbose=0)
    return acc

# =========================
# Run experiments
# =========================
results = []

for dataset_name in ["CIFAR10", "CIFAR100"]:
    x_train, y_train, x_test, y_test, num_classes = load_dataset(dataset_name)

    for temp in TEMPS:
        for alpha in ALPHAS:
            for p in LABEL_PERCENTS:

                xs, ys = get_subset(x_train, y_train, p)

                train_ds = make_ds(xs, ys, True)
                test_ds = make_ds(x_test, y_test, False)

                model = get_model()
                train(model, train_ds, temp, alpha)

                acc = evaluate(model, train_ds, test_ds, num_classes)

                print(dataset_name, "| APW-FIXED |", p,
                      "| temp=", temp, "| alpha=", alpha, "|", acc)

                results.append([
                    dataset_name,
                    "APW-FIXED",
                    p,
                    temp,
                    alpha,
                    acc
                ])

# =========================
# GROUPED RESULTS
# =========================
grouped = defaultdict(list)

for r in results:
    dataset, method, p, temp, alpha, acc = r
    grouped[(dataset, p)].append((temp, alpha, acc))

print("\n\n===== GROUPED RESULTS =====\n")

for (dataset, p) in sorted(grouped.keys()):
    print(f"=== {dataset} | Label Percent = {p} ===")

    rows = sorted(grouped[(dataset, p)],
                  key=lambda x: x[2],
                  reverse=True)

    for temp, alpha, acc in rows:
        print(f"Temp={temp} | Alpha={alpha} | Acc={acc:.4f}")

    print()

# =========================
# BEST CONFIG SUMMARY
# =========================
print("\n===== BEST CONFIG PER SETTING =====\n")

for (dataset, p) in sorted(grouped.keys()):
    best = max(grouped[(dataset, p)], key=lambda x: x[2])
    temp, alpha, acc = best

    print(f"{dataset} | {p} | Temp={temp} | Alpha={alpha} | Acc={acc:.4f}")

CIFAR10 | APW-FIXED | 0.01 | temp= 0.05 | alpha= 0.5 | 0.16110000014305115
CIFAR10 | APW-FIXED | 0.05 | temp= 0.05 | alpha= 0.5 | 0.2685999870300293
CIFAR10 | APW-FIXED | 0.1 | temp= 0.05 | alpha= 0.5 | 0.28519999980926514
CIFAR10 | APW-FIXED | 0.5 | temp= 0.05 | alpha= 0.5 | 0.4553999900817871
CIFAR10 | APW-FIXED | 0.01 | temp= 0.05 | alpha= 1.0 | 0.17739999294281006
CIFAR10 | APW-FIXED | 0.05 | temp= 0.05 | alpha= 1.0 | 0.2401999980211258
CIFAR10 | APW-FIXED | 0.1 | temp= 0.05 | alpha= 1.0 | 0.2775000035762787
CIFAR10 | APW-FIXED | 0.5 | temp= 0.05 | alpha= 1.0 | 0.4196000099182129
CIFAR10 | APW-FIXED | 0.01 | temp= 0.1 | alpha= 0.5 | 0.17739999294281006
CIFAR10 | APW-FIXED | 0.05 | temp= 0.1 | alpha= 0.5 | 0.2709999978542328
CIFAR10 | APW-FIXED | 0.1 | temp= 0.1 | alpha= 0.5 | 0.31209999322891235
CIFAR10 | APW-FIXED | 0.5 | temp= 0.1 | alpha= 0.5 | 0.46140000224113464
CIFAR10 | APW-FIXED | 0.01 | temp= 0.1 | alpha= 1.0 | 0.1623000055551529
CIFAR10 | APW-FIXED | 0.05 | temp= 0.1 | al